In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/Pavan-gs/ai-services-bbsr/refs/heads/main/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (7032, 19)
y shape: (7032,)


In [3]:
numeric_cols = X.select_dtypes(include='number').columns
categorical_cols = X.select_dtypes(exclude='number').columns

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training rows:', len(X_train))
print('Test rows:', len(X_test))

Training rows: 5625
Test rows: 1407


In [5]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

models

{'Decision Tree': DecisionTreeClassifier(random_state=42),
 'Random Forest': RandomForestClassifier(random_state=42),
 'SVM': SVC(probability=True, random_state=42)}

In [6]:
results = []
fitted_pipelines = {}

for name, classifier in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', classifier)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

    fitted_pipelines[name] = pipeline

In [7]:
results_df = pd.DataFrame(results)
results_df.round(4)

,Model,Accuracy,Precision,Recall,ROC-AUC
0,Decision Tree,0.7306,0.4934,0.4973,0.6565
1,Random Forest,0.7875,0.6307,0.4840,0.8137
2,SVM,0.7918,0.6401,0.4947,0.7886


In [8]:
ranked_results = results_df.sort_values(
    by=['Recall', 'ROC-AUC'], ascending=False
).reset_index(drop=True)

champion_name = ranked_results.loc[0, 'Model']
champion_pipeline = fitted_pipelines[champion_name]

print('Champion:', champion_name)
ranked_results.round(4)

Champion: Decision Tree


,Model,Accuracy,Precision,Recall,ROC-AUC
0,Decision Tree,0.7306,0.4934,0.4973,0.6565
1,SVM,0.7918,0.6401,0.4947,0.7886
2,Random Forest,0.7875,0.6307,0.4840,0.8137


In [9]:
joblib.dump(champion_pipeline, '../models/champion_model.joblib')
print('Saved:', '../models/champion_model.joblib')

Saved: ../models/champion_model.joblib


In [10]:
print('DAY 2 MODEL COMPARISON')
print('-' * 60)
for _, row in results_df.iterrows():
    print(
        f"{row['Model']}: "
        f"Accuracy={row['Accuracy']:.4f}, "
        f"Precision={row['Precision']:.4f}, "
        f"Recall={row['Recall']:.4f}, "
        f"ROC-AUC={row['ROC-AUC']:.4f}"
    )

winner = ranked_results.iloc[0]
print('\nChampion:', champion_name)
print(f"Reason: highest churn recall ({winner['Recall']:.4f}); ROC-AUC={winner['ROC-AUC']:.4f}.")

DAY 2 MODEL COMPARISON
------------------------------------------------------------
Decision Tree: Accuracy=0.7306, Precision=0.4934, Recall=0.4973, ROC-AUC=0.6565
Random Forest: Accuracy=0.7875, Precision=0.6307, Recall=0.4840, ROC-AUC=0.8137
SVM: Accuracy=0.7918, Precision=0.6401, Recall=0.4947, ROC-AUC=0.7886

Champion: Decision Tree
Reason: highest churn recall (0.4973); ROC-AUC=0.6565.
